# AIDS Survival Analysis
## Dataset Analysis and Statistical Summary

This notebook performs comprehensive exploratory data analysis and statistical testing on the AIDS Classification dataset.

**Sections:**
1. Environment Setup
2. Data Loading & Exploration
3. Feature Engineering
4. Data Splitting Strategy
5. Statistical Analysis
6. Summary Tables Generation

## 1. Environment Setup
### Import Required Libraries and Configure Visualization Settings

In [1]:
# Data manipulation and numerical computing
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')

#Set high-resolution output for all figures (1000 DPI)
plt.rcParams["figure.dpi"] = 1000
plt.rcParams["savefig.dpi"] = 1000

# Machine learning and survival analysis
from sklearn.model_selection import train_test_split
from sksurv.metrics import concordance_index_censored, brier_score, cumulative_dynamic_auc
from sksurv.linear_model import CoxPHSurvivalAnalysis

# Statistical testing
from scipy.stats import f_oneway, chi2_contingency

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

## 2. Data Loading & Exploration
### Load the AIDS Classification Dataset

In [2]:
# Load dataset from CSV file
df = pd.read_csv("AIDS_Classification_50000.csv")

### Preview First Few Rows of the Dataset

In [3]:
# Display the first 5 rows to understand the data structure
df.head()

,time,trt,age,wtkg,hemo,homo,drugs,karnof,oprior,z30,...,str2,strat,symptom,treat,offtrt,cd40,cd420,cd80,cd820,infected
0,1073,1,37,79.46339,0,1,0,100,0,1,...,1,2,0,1,0,322,469,882,754,1
1,324,0,33,73.02314,0,1,0,90,0,1,...,1,3,1,1,1,168,575,1035,1525,1
2,495,1,43,69.47793,0,1,0,100,0,1,...,1,1,0,0,0,377,333,1147,1088,1
3,1201,3,42,89.15934,0,1,0,100,1,1,...,1,3,0,0,0,238,324,775,1019,1
4,934,0,37,137.46581,0,1,0,100,0,0,...,0,3,0,0,1,500,443,1601,849,0


### Dataset Information
Check data types, non-null counts, and memory usage

In [4]:
# Display comprehensive information about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 23 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   time      50000 non-null  int64  
 1   trt       50000 non-null  int64  
 2   age       50000 non-null  int64  
 3   wtkg      50000 non-null  float64
 4   hemo      50000 non-null  int64  
 5   homo      50000 non-null  int64  
 6   drugs     50000 non-null  int64  
 7   karnof    50000 non-null  int64  
 8   oprior    50000 non-null  int64  
 9   z30       50000 non-null  int64  
 10  preanti   50000 non-null  int64  
 11  race      50000 non-null  int64  
 12  gender    50000 non-null  int64  
 13  str2      50000 non-null  int64  
 14  strat     50000 non-null  int64  
 15  symptom   50000 non-null  int64  
 16  treat     50000 non-null  int64  
 17  offtrt    50000 non-null  int64  
 18  cd40      50000 non-null  int64  
 19  cd420     50000 non-null  int64  
 20  cd80      50000 non-null  in

### Check for Missing Values

In [5]:
# Count missing values for each feature
df.isnull().sum().T

time        0
trt         0
age         0
wtkg        0
hemo        0
homo        0
drugs       0
karnof      0
oprior      0
z30         0
preanti     0
race        0
gender      0
str2        0
strat       0
symptom     0
treat       0
offtrt      0
cd40        0
cd420       0
cd80        0
cd820       0
infected    0
dtype: int64

## 3. Feature Engineering
### Prepare Features and Target Variables

- **Target Variables:**
  - `time`: Survival time in days
  - `infected`: Event indicator (1=event occurred, 0=censored)
- **Features:** All other columns excluding time and infected

In [6]:
# Define column names for time-to-event and event indicator
time_col = "time"
event_col = "infected"

# Separate features from target variables
X = df.drop(columns=[time_col, event_col])
y_time = df[time_col].values
y_event = df[event_col].values.astype(bool)  # Convert to boolean for sksurv compatibility

### Compute Time Horizons for Evaluation
Calculate quartile-based time points (25th, 50th, 75th percentiles) from observed event times.

These horizons will be used for time-dependent evaluation metrics.

In [7]:
# Extract time and event data
t = df[time_col].values
e = df[event_col].values

# Define horizons as quartiles (25%, 50%, 75%)
horizons = [0.25, 0.5, 0.75]

# Compute time points at these quartiles from OBSERVED events only (e == 1)
times = np.quantile(t[e == 1], horizons).tolist()
print(f"Time horizons (days): {times}")

Time horizons (days): [496.0, 992.0, 1127.0]


### Identify Categorical and Numerical Features

- **Categorical:** Features with less than 10 unique values
- **Numerical:** All other features

In [8]:
# Identify categorical columns (features with less than 10 unique values)
cat_cols = [col for col in X.columns if X[col].nunique() < 10]

# Remaining columns are considered numerical
num_cols = [col for col in X.columns if col not in cat_cols]

print(f"Categorical features ({len(cat_cols)}): {cat_cols}")
print(f"Numerical features ({len(num_cols)}): {num_cols}")

Categorical features (13): ['trt', 'hemo', 'homo', 'drugs', 'oprior', 'z30', 'race', 'gender', 'str2', 'strat', 'symptom', 'treat', 'offtrt']
Numerical features (8): ['age', 'wtkg', 'karnof', 'preanti', 'cd40', 'cd420', 'cd80', 'cd820']


## 4. Data Splitting Strategy
### Train-Test-Validation Split

**Split Ratios:**
- Training: 70% (~35,000 samples)
- Testing: 21% (~10,500 samples before filtering)
- Validation: 9% (~4,500 samples before filtering)

**Important:** Samples with time values exceeding the maximum training time are removed from test/validation sets.
This ensures proper model evaluation without extrapolation beyond the training distribution.

In [9]:
# First split: separate training set (70%) from temporary set (30%)
x_train, x_temp, t_train, t_temp, e_train, e_temp = train_test_split(
    X, y_time, y_event, 
    test_size=0.30, 
    random_state=42, 
    stratify=e  # Stratify by event to maintain event ratio across splits
)

# Second split: divide temporary set into test (70% of 30%) and validation (30% of 30%)
x_test, x_val, t_test, t_val, e_test, e_val = train_test_split(
    x_temp, t_temp, e_temp, 
    test_size=0.30, 
    random_state=42, 
    stratify=e_temp  # Stratify by event
)

# Remove samples from test set where time exceeds max training time
# This prevents evaluation on times beyond what the model was trained on
mask_test = t_test < t_train.max()
x_test = x_test[mask_test]
t_test = t_test[mask_test]
e_test = e_test[mask_test]

# Similarly, remove samples from validation set where time exceeds max training time
mask_val = t_val < t_train.max()
x_val = x_val[mask_val]
t_val = t_val[mask_val]
e_val = e_val[mask_val]

# Display final sample sizes after filtering
print(f"Train shape: {x_train.shape}")
print(f"Test shape: {x_test.shape}")
print(f"Validation shape: {x_val.shape}")

Train shape: (35000, 21)
Test shape: (10480, 21)
Validation shape: (4495, 21)


## 5. Statistical Analysis
### Comparing Distributions Across Cohorts

We perform statistical tests to ensure that train, test, and validation sets are drawn from the same distribution.
This validates the quality of our data splitting strategy.

### 5.1 ANOVA F-test for Numerical Variables

Test whether numerical features have similar distributions across all cohorts (overall, train, test, validation).

**Null Hypothesis:** All cohorts have the same mean for each numerical feature.

**High p-values (>0.05) indicate no significant difference between cohorts (desired outcome).**

In [10]:
# Perform one-way ANOVA for each numerical feature
p_val = []
for col in num_cols:
    # Compare the feature across all four cohorts
    f_stat, p_value = f_oneway(
        df[col], 
        x_train[col], 
        x_test[col], 
        x_val[col]
    )
    p_val.append(round(p_value, 3))

# Also test the time variable distribution across cohorts
f_stat, p_value = f_oneway(df[time_col], t_train, t_test, t_val)
p_val.append(round(p_value, 3))

### 5.2 Numerical Variables Summary Table

Display mean (μ) and standard deviation (σ) for each numerical feature across all cohorts, along with ANOVA p-values.

This table helps verify that the cohorts have similar distributions.

In [11]:
# Calculate descriptive statistics for each cohort
train_stats = x_train[num_cols].describe().T.round(3)[["mean", "std"]]
test_stats = x_test[num_cols].describe().T.round(3)[["mean", "std"]]
val_stats = x_val[num_cols].describe().T.round(3)[["mean", "std"]]
over_all_stats = df[num_cols].describe().T.round(3)[["mean", "std"]]

# Format statistics as "mean(std)" strings
train_data = []
test_data = []
val_data = []
overall_data = []
variable = []

for i in range(len(num_cols)):
    train_mean = train_stats["mean"].iloc[i]
    train_std = train_stats["std"].iloc[i]
    test_mean = test_stats["mean"].iloc[i]
    test_std = test_stats["std"].iloc[i]
    val_mean = val_stats["mean"].iloc[i]
    val_std = val_stats["std"].iloc[i]
    overall_mean = over_all_stats["mean"].iloc[i]
    overall_std = over_all_stats["std"].iloc[i]
    
    train_data.append(f"{train_mean}({train_std})")
    test_data.append(f"{test_mean}({test_std})")
    val_data.append(f"{val_mean}({val_std})")
    overall_data.append(f"{overall_mean}({overall_std})")
    variable.append(num_cols[i])
    
# Add time variable statistics
variable.append("Time (days)")
overall_data.append(f"{y_time.mean().round(3)}({y_time.std().round(3)})")
train_data.append(f"{t_train.mean().round(3)}({t_train.std().round(3)})")
test_data.append(f"{t_test.mean().round(3)}({t_test.std().round(3)})")
val_data.append(f"{t_val.mean().round(3)}({t_val.std().round(3)})")

# Create summary DataFrame
descriptives = pd.DataFrame(
    {
        "Variable": variable,
        "Overall μ (σ)": overall_data,
        "Training Cohort μ (σ)": train_data,
        "Testing Cohort μ (σ)": test_data,
        "Validation Cohort μ (σ)": val_data,
        "P-value": p_val,
    }
)

# Export to CSV
descriptives.to_csv("numeric_variable_summary.csv", index=False)
display(descriptives)

,Variable,Overall μ (σ),Training Cohort μ (σ),Testing Cohort μ (σ),Validation Cohort μ (σ),P-value
0,age,34.164(7.091),34.192(7.106),34.079(7.045),34.151(7.092),0.555
1,wtkg,75.862(12.029),75.816(12.042),76.017(12.018),75.861(11.946),0.521
2,karnof,96.832(5.092),96.809(5.109),96.842(5.087),96.977(4.967),0.223
3,preanti,318.16(402.933),317.96(402.096),320.637(406.15),313.961(402.078),0.827
4,cd40,319.08(102.526),319.5(102.657),317.906(101.991),318.584(102.725),0.559
5,cd420,438.09(144.807),437.887(144.638),438.452(144.291),438.349(146.968),0.986
6,cd80,1045.936(488.617),1044.774(487.557),1050.46(493.536),1043.96(484.5),0.760
7,cd820,905.938(339.708),904.563(340.496),908.487(339.928),910.464(333.515),0.581
8,Time (days),877.37(307.286),876.748(307.733),879.757(305.862),874.678(306.798),0.774


### 5.3 Chi-Square Test for Categorical Variables

Test the independence of categorical feature distributions across cohorts.

**Null Hypothesis:** The distribution of category levels is independent of the cohort.

**High p-values (>0.05) indicate no significant difference between cohorts (desired outcome).**

In [14]:
# Perform chi-square test for each categorical feature including event_col
cat_pvalues = []
cols_to_test = cat_cols + [event_col]
for i in cols_to_test:
    # Create contingency table of value counts across all cohorts
    if i == event_col:
        contingency_table = pd.concat([
            df[event_col].value_counts(),
            pd.Series(e_train).value_counts(),
            pd.Series(e_test).value_counts(),
            pd.Series(e_val).value_counts()
        ], axis=1)
    else:
        contingency_table = pd.concat([
            df[i].value_counts(),
            x_train[i].value_counts(),
            x_test[i].value_counts(),
            x_val[i].value_counts()
        ], axis=1)
    
    # Perform chi-square test
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    cat_pvalues.append(round(p_value, 3))

# Store p-values in a Series for easy lookup
cat_pvalues = pd.Series(cat_pvalues, index=cols_to_test)


### 5.4 Categorical Variables Summary Table

Display counts and percentages for each category level across all cohorts, along with chi-square p-values.

This table helps verify that the cohorts have similar categorical distributions.

In [29]:
# List to store individual dataframes for each categorical variable
all_summaries = []

# Iterate through each categorical column and the event column
for col in (cat_cols + [event_col]):
# Calculate counts and percentages for each cohort
    if col == event_col:
        total_counts = df[col].value_counts()
        total_percentages = df[col].value_counts(normalize=True) * 100
        train_counts = pd.Series(e_train).value_counts()
        train_percentages = pd.Series(e_train).value_counts(normalize=True) * 100
        test_counts = pd.Series(e_test).value_counts()
        test_percentages = pd.Series(e_test).value_counts(normalize=True) * 100
        val_counts = pd.Series(e_val).value_counts()
        val_percentages = pd.Series(e_val).value_counts(normalize=True) * 100
    else:
        total_counts = df[col].value_counts()
        total_percentages = df[col].value_counts(normalize=True) * 100
        train_counts = x_train[col].value_counts()
        train_percentages = x_train[col].value_counts(normalize=True) * 100
        test_counts = x_test[col].value_counts()
        test_percentages = x_test[col].value_counts(normalize=True) * 100
        val_counts = x_val[col].value_counts()
        val_percentages = x_val[col].value_counts(normalize=True) * 100

    # Create the combined DataFrame for this variable
    summary_df = pd.concat(
        [
            total_counts, total_percentages,
            train_counts, train_percentages,
            test_counts, test_percentages,
            val_counts, val_percentages
        ],
        axis=1,
        keys=[
            'Total Count', 'Total Percentage',
            'Train Count', 'Train Percentage',
            'Test Count', 'Test Percentage',
            'Val Count', 'Val Percentage'
        ]
    )

    # Add a column "Variable" to identify which categorical variable these rows belong to
    summary_df.insert(0, 'Variable', col)

    # Rename the index to represent the specific categories (levels)
    summary_df.index.name = 'Level'

    # Add the p-value for this categorical variable
    summary_df['P-value'] = cat_pvalues.get(col, None)

    # Format percentages to two decimal places
    summary_df[['Total Percentage', 'Train Percentage', 'Test Percentage', 'Val Percentage']] = \
        summary_df[['Total Percentage', 'Train Percentage', 'Test Percentage', 'Val Percentage']].round(2).astype(str) + '%'

    # Append to list
    all_summaries.append(summary_df)

# Merge all tables into one
final_summary_table = pd.concat(all_summaries)

# Reset index to make 'Level' a regular column
final_summary_table.reset_index(inplace=True)

# Reorder columns so Variable is first, then Level, then the rest
cols = final_summary_table.columns.tolist()
cols = ['Variable', 'Level'] + [c for c in cols if c not in ['Variable', 'Level']]
final_summary_table = final_summary_table[cols].sort_values(by=["Variable", "Level"]).reset_index(drop=True)

# Export to CSV
final_summary_table.to_csv("categorical_variable_summary.csv", index=False)
display(final_summary_table)

        

,Variable,Level,Total Count,Total Percentage,Train Count,Train Percentage,Test Count,Test Percentage,Val Count,Val Percentage,P-value
0,drugs,0,43389,86.78%,30382,86.81%,9075,86.59%,3909,86.96%,0.926
1,drugs,1,6611,13.22%,4618,13.19%,1405,13.41%,586,13.04%,0.926
2,gender,0,7165,14.33%,5036,14.39%,1488,14.2%,640,14.24%,0.965
3,gender,1,42835,85.67%,29964,85.61%,8992,85.8%,3855,85.76%,0.965
4,hemo,0,48326,96.65%,33859,96.74%,10094,96.32%,4351,96.8%,0.190
5,hemo,1,1674,3.35%,1141,3.26%,386,3.68%,144,3.2%,0.190
6,homo,0,17323,34.65%,12120,34.63%,3627,34.61%,1567,34.86%,0.991
7,homo,1,32677,65.35%,22880,65.37%,6853,65.39%,2928,65.14%,0.991
8,infected,0,34494,68.99%,24146,68.99%,7228,68.97%,3100,68.97%,1.000
9,infected,1,15506,31.01%,10854,31.01%,3252,31.03%,1395,31.03%,1.000


## 6. Exploratory Visualizations (Optional)

The following cells contain commented-out visualization code. Uncomment to generate plots:
- Boxplot of survival times
- Pie chart of event proportions  
- Histogram of time distribution by event status

In [ ]:
# Boxplot showing distribution of survival times
# plt.figure(figsize=(5, 2))
# sns.boxplot(x=df[time_col])
# plt.xlabel("Time in days".upper())
# plt.show()

In [ ]:
# Pie chart showing proportion of censored vs. infected cases
# plt.figure(figsize=(5, 5))
# labels = ["Censored", "Event"]
# colors = sns.color_palette("pastel")
# plot_data = df[event_col].value_counts()
# plt.pie(x=plot_data.values, autopct="%.2f%%", colors=colors, labels=labels)
# plt.title("Infected".upper())
# plt.show()

In [ ]:
# Histogram showing time distribution separated by event status
# plt.figure(figsize=(4, 4))
# sns.histplot(data=df, x=time_col, hue=event_col, kde=True)
# plt.title("Time distribution of infections".upper(), size=10)
# plt.ylabel("Frequency")
# plt.xlabel("Time")
# plt.tick_params(axis="y", labelsize=5)
# plt.tick_params(axis="x", labelsize=5)
# plt.legend(["Censored", "Infected"])
# plt.show()